# TCGA-BRCA WFCM

The target cohort contains 819 primary-tumor samples with PAM50 counts Basal=139, Her2=67, LumA=421, LumB=192. k=4 is fixed for this four-subtype comparison; WFCM selects m by minimum refined numerical NLL. Four observed initialization seeds are compared by NLL, not by PAM50 performance. Bootstrap holds selected k,m fixed and warm-starts from the observed fit. 

### Required data
Recommended exact-reference mode: provide `data/TCGA_BRCA_prepared_pcs.csv` with a unique sample-ID index, PC1–PC5 and PAM50. This prepared dataset must be provided separately; it is not embedded.

Alternative raw mode includes all preprocessing code: configure a local STAR TPM matrix (`Ensembl_ID` plus sample columns) and the clinical source below. It selects one primary tumor per patient, matches clinical annotations at sample level, keeps the four PAM50 groups, chooses 5,000 variable genes and fits five PCs directly on the supplied values (no extra log transform/gene standardization/PC scaling).

### Numerical method and reference
Defensive proposal: 20% data GMM, 70% isotropic Gaussian coverage, 10% Student-t(df=5). Integration uses 32,768 scrambled Sobol points with full proposal densities; an independent 131,072-point cloud checks m ranking. A separate million-draw Gaussian-envelope audit precedes bootstrap. Sobol integration is a numerical extension: it does not automatically inherit an iid-importance-sampling theorem. Bootstrap observations themselves are independent analytic-rejection draws; proposals/clouds are rebuilt per replicate. Scalar SDs use B and covariances B−1; full five-dimensional center regions use chi-square df=5. The displayed PC1–PC2 **marginal** regions use df=2 and are not the projection of the full five-dimensional region.

Reference verified result: selected m≈2.00845438694, initialization seed 42, fitting NLL≈18324.697727, independent bounded-envelope NLL≈18324.836271; B=100. K-means/ordinary FCM are descriptive comparisons; ordinary FCM selects its m by profiling the equal-weight likelihood at its fitted centers. The multi-start wrapper, RQMC integration and fixed-cohort/PC conditioning must be stated when reporting this application. They do not prove global optimality or selection-adjusted confidence coverage. Supplementary t-SNE does not refit clustering; UMAP panels are optional and require `umap-learn`.


## Setup
This is **one self-contained code file**: all WFCM fitting, integration, inference and plotting functions are embedded. **Datasets are separate inputs**, not embedded in this notebook. Start a fresh Python kernel, configure the data path, then run every cell in order. 

Tested Python: 3.12.9. A matching environment can be installed with:
```python
%pip install numpy==2.5.4 scipy==1.18.1 pandas==3.1.0 scikit-learn==1.9.1 matplotlib==3.11.2
```
Run the installation command separately if needed; this notebook does not automatically modify your environment. Full inference uses B=100 and can be slow. Decreasing B or the integration budget is a diagnostic run, not reproduction of the reference result.


In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
os.environ.setdefault('OMP_MAX_ACTIVE_LEVELS','1')
os.environ.setdefault('LOKY_MAX_CPU_COUNT','1')


In [ ]:
from pathlib import Path
import json
import hashlib
import platform
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linear_sum_assignment
from scipy.special import logsumexp, xlogy
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SUBTYPES = ['Basal', 'Her2', 'LumA', 'LumB']


# Portable data lookup; set WFCM_DATA_DIR or edit the input setting below.
import os, hashlib, platform
from pathlib import Path

def locate_data(filename, required=True):
    preferred=Path(os.environ.get('WFCM_DATA_DIR','data'))/filename
    candidates=[preferred,Path.cwd()/'data'/filename,Path.cwd().parent/'data'/filename,Path.cwd()/filename]
    for candidate in candidates:
        if candidate.is_file():return candidate.resolve()
    if required:
        raise FileNotFoundError(f'Required dataset {filename!r} is not bundled. Put it in data/, set WFCM_DATA_DIR, or edit the input path. No substitute data will be used.')
    return preferred.resolve()

# Save every displayed figure, including plots without an explicit savefig call.
# Individual records are written only to local output files after execution.
import matplotlib.pyplot as plt
_base_show=getattr(plt.show,'_wfcm_base_show',plt.show)
_saved_figure_files=[]
def _show_and_save(*args,**kwargs):
    directory=Path(OUTPUT_DIR)/'figures';directory.mkdir(parents=True,exist_ok=True)
    for number in plt.get_fignums():
        figure=plt.figure(number)
        if not getattr(figure,'_wfcm_exported',False):
            file=directory/f'figure_{len(_saved_figure_files)+1:02d}.png'
            figure.savefig(file,dpi=180,bbox_inches='tight');figure._wfcm_exported=True
            _saved_figure_files.append(str(file))
    return _base_show(*args,**kwargs)
_show_and_save._wfcm_base_show=_base_show
plt.show=_show_and_save


## 1. Settings and dataset formats


In [ ]:
INPUT_MODE = 'prepared'                     # raw / prepared; both run from a fresh kernel
EXPR_PATH = locate_data('TCGA-BRCA.star_tpm.tsv.gz',required=False)
CLINICAL_SOURCE = ('https://tcga.xenahubs.net/download/'
                   'TCGA.BRCA.sampleMap/BRCA_clinicalMatrix')
PREPARED_PATH = locate_data('TCGA_BRCA_prepared_pcs.csv',required=False)
STANDARDIZE_GENES = False              # Confirmed: PCA directly on the selected genes.
N_HVG, N_PC, K = 5000, 5, 4
EXPECTED_N = 819
EXPECTED_COUNTS = {'Basal': 139, 'Her2': 67, 'LumA': 421, 'LumB': 192}
STRICT_COHORT = True                   # Prevent accidental changes in cohort.
MODE = 'select'                        # fixed / select; decide before evaluation.
FIXED_M = 2                            # only used if MODE='fixed'
M_GRID = np.geomspace(1.5, 2.5, 8)        # Candidate m-grid for NLL selection.
WFCM_ENGINE = 'manuscript_defensive_rqmc'              # this notebook uses the manuscript implementation
WXB_MIN = None                        # Retained metadata only; no WFCM XB cutoff.
FCM_SEEDS = [321, 42, 7, 199]  
WFCM_SEEDS = [321, 42, 7, 199]             # multiple restarts improves stability
NUM_IS = 32768                        # 2**15 scrambled Sobol integration points
VALIDATION_NUM_IS = 131072              # independent 2**17-point integration check
PC_SENSITIVITY_NUM_IS = 131072           # Same count for fitting and independent checking
PROPOSAL_COMPONENTS = K                 # unsupervised data GMM component count
NORMALIZER_NLL_TOL = 2.0                # numerical accuracy diagnostic, not a fitting tolerance
IS_SEED, VALIDATION_SEED = 674, 675
# Two candidates exhausted 200 outer steps; retain the stopping tolerances.
MAX_OUTER, INNER_MAXITER = 2000, 1000
K_POST = 1000
B, ALPHA = 100, 0.05
OUTPUT_DIR = Path('results_breast').resolve()
BOOTSTRAP_SEED = 999
EXPECTED_PREPARED_HASH='f6619a55aecfdfce6612eb9f945efec06a50f01b9c5551fa79eb6152efced26d'
STRICT_REFERENCE_INPUT=True  # set False only for an explicitly different analysis
EXPECTED_LABEL_HASH='3ee337ca7fd1187a31ac2a4c7cf8e8e88d13ad9db77a04d9c6cb6e7c28cad757'

RUN_PC_SENSITIVITY = False  # Set True to run Appendix H.4 after the main analysis.


## 2. Shared data preparation
PAM50 defines the four-subtype evaluation cohort; subtype identities are never
used for feature selection, fitting, initialization, or choosing m/seeds.

In [ ]:
def kaiser_guttman(features):
    """Correlation-PCA diagnostic on the complete pre-PCA feature matrix.

    Sample-SD scaling (ddof=1) makes eigenvalues comparable to one.
    This diagnostic does not change the features used for model fitting.
    """
    x = np.asarray(features, dtype=float)
    if x.ndim != 2 or min(x.shape) < 2 or not np.isfinite(x).all():
        raise ValueError('Provide a finite samples-by-features matrix with at least two rows/columns.')
    sd = x.std(axis=0, ddof=1)
    keep = sd > 0
    if not keep.any():
        raise ValueError('No nonconstant features.')
    z = (x[:, keep] - x[:, keep].mean(axis=0)) / sd[keep]
    # Singular values avoid constructing a large genes-by-genes matrix.
    eigenvalues = np.linalg.svd(z, compute_uv=False) ** 2 / (len(z) - 1)
    # Include the zero eigenvalues when features outnumber samples.
    eigenvalues = np.pad(eigenvalues, (0, max(0, int(keep.sum()) - len(eigenvalues))))
    table = pd.DataFrame({'PC': np.arange(1, len(eigenvalues) + 1),
                          'correlation_eigenvalue': eigenvalues,
                          'variance_fraction': eigenvalues / eigenvalues.sum(),
                          'retain_KG': eigenvalues > 1.0})
    table['cumulative_variance'] = table.variance_fraction.cumsum()
    print('Kaiser–Guttman PCs (eigenvalue > 1):', int(table.retain_KG.sum()))
    print('Constant features excluded:', int((~keep).sum()))
    return table


In [ ]:
def load_raw_features(expr_path, clinical_source, report_dir):
    if not Path(expr_path).is_file():
        raise FileNotFoundError(f'Set EXPR_PATH to your downloaded STAR TPM matrix: {expr_path}')
    expr = pd.read_csv(expr_path, sep='\t', low_memory=False)
    clinical = pd.read_csv(clinical_source, sep='\t', low_memory=False)
    for col in ['sampleID', 'PAM50Call_RNAseq']:
        if col not in clinical:
            raise ValueError(f'Clinical table missing {col}')
    if 'Ensembl_ID' not in expr:
        raise ValueError('Expected expression gene column Ensembl_ID.')
    genes = expr.set_index('Ensembl_ID')
    if not genes.index.is_unique or not genes.columns.is_unique:
        raise ValueError('Duplicate expression identifiers; inspect before proceeding.')
    info = pd.DataFrame({'sample_id': genes.columns})
    if not info.sample_id.str.match(r'^TCGA-[^-]+-[^-]+-\d{2}').all():
        raise ValueError('Unexpected TCGA sample barcode format.')
    info['patient_id'] = info.sample_id.str[:12]
    info['sample_type'] = info.sample_id.str.split('-').str[3].str[:2]
    primary = (info.loc[info.sample_type.eq('01')]
               .sort_values(['patient_id', 'sample_id'])
               .drop_duplicates('patient_id', keep='first'))
    pam = clinical[['sampleID', 'PAM50Call_RNAseq']].rename(
        columns={'sampleID': 'clinical_sample_id', 'PAM50Call_RNAseq': 'PAM50'}).copy()
    # Expression uses vial IDs (...-01A); the clinical table uses sample IDs
    # (...-01). Match at the clinical table's sample level, not patient level.
    primary = primary.copy()
    primary['sample_key'] = primary.sample_id.str[:15]
    pam['sample_key'] = pam.clinical_sample_id.str[:15]
    pam['patient_id'] = pam.clinical_sample_id.str[:12]
    selected = pam.loc[pam.sample_key.isin(primary.sample_key)].copy()
    conflicts = selected.groupby('sample_key').PAM50.nunique(dropna=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    if (conflicts > 1).any():
        bad = selected.loc[selected.sample_key.isin(conflicts[conflicts > 1].index)]
        report = report_dir / 'conflicting_selected_sample_annotations.csv'
        bad.to_csv(report, index=False)
        raise ValueError('Conflicting calls for the SAME selected tumor sample; '
                         f'inspect {report.resolve()}')
    # Consistent repeated records are safe to collapse. Prefer a known call to
    # a missing duplicate, but never substitute an annotation from another sample.
    annotation = (selected.assign(_missing=selected.PAM50.isna())
                  .sort_values('_missing', kind='stable')
                  .drop_duplicates('sample_key', keep='first'))
    info = primary.merge(annotation[['sample_key', 'PAM50']], on='sample_key',
                         how='left', validate='one_to_one')
    legacy = pam.drop_duplicates('patient_id', keep='first').set_index('patient_id').PAM50
    info['previous_patient_first_PAM50'] = info.patient_id.map(legacy)
    info['label_changed_vs_patient_first'] = (
        info.PAM50.fillna('<missing>') != info.previous_patient_first_PAM50.fillna('<missing>'))
    info['annotation_status'] = np.where(info.PAM50.notna(), 'sample_matched',
        np.where(info.sample_key.isin(annotation.sample_key), 'missing_PAM50', 'unmatched_sample'))
    info['included_in_analysis'] = info.PAM50.isin(SUBTYPES)
    report = report_dir / 'sample_annotation_audit.csv'
    info.to_csv(report, index=False)
    print('Sample annotation matching:', info.annotation_status.value_counts().to_dict())
    print('Labels differing from the old patient-first merge:',
          int(info.label_changed_vs_patient_first.sum()))
    print('Matching report:', report.resolve())
    info = info.loc[info.PAM50.isin(SUBTYPES)].set_index('sample_id')
    X_gene = genes.loc[:, info.index].T.copy()
    info = info.loc[X_gene.index]
    y = info.PAM50.copy()
    X_gene = X_gene.replace([np.inf, -np.inf], np.nan)
    X_gene = X_gene.loc[:, X_gene.notna().all(axis=0)]
    var = X_gene.var(axis=0)
    X_gene = X_gene.loc[:, var > 1e-8]
    chosen = X_gene.var(axis=0).sort_values(ascending=False).head(N_HVG).index
    hvg = X_gene.loc[:, chosen]
    return hvg, y, info, list(chosen)


def pca_features(hvg, standardize_genes):
    if standardize_genes is not True and standardize_genes is not False:
        raise ValueError('Set STANDARDIZE_GENES to True or False.')
    return StandardScaler().fit_transform(hvg) if standardize_genes else hvg.to_numpy()


def load_raw_data(expr_path, clinical_source, standardize_genes):
    report_dir = Path(globals().get('OUTPUT_DIR', 'TCGA_BRCA_comparison_results'))
    hvg, y, info, chosen = load_raw_features(expr_path, clinical_source, report_dir)
    if hvg.shape[1] < N_PC:
        raise ValueError('Too few valid genes for requested PCA.')
    features = pca_features(hvg, standardize_genes)
    global kg_tcga
    kg_tcga = kaiser_guttman(hvg.to_numpy())
    kg_tcga.to_csv(report_dir / 'kaiser_guttman.csv', index=False)
    display(kg_tcga.head(20))
    pca = PCA(n_components=N_PC, random_state=123)
    pcs = pca.fit_transform(features)
    return pcs, y, info, pca, chosen


def check_input(X, y):
    if not isinstance(y, pd.Series):
        raise ValueError('Provide labels as a Series indexed by patient/sample IDs to preserve alignment.')
    if isinstance(X, pd.DataFrame) and not X.index.equals(y.index):
        raise ValueError('X_pca and labels indices differ; align explicitly first.')
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape != (len(y), N_PC) or not np.isfinite(X).all():
        raise ValueError(f'Expected finite patients x {N_PC} PC matrix matching labels.')
    if not y.index.is_unique or y.isna().any() or set(y) != set(SUBTYPES):
        raise ValueError('Invalid sample IDs or PAM50 labels.')
    if STRICT_COHORT and (len(y) != EXPECTED_N or y.value_counts().to_dict() != EXPECTED_COUNTS):
        raise ValueError('Cohort changed. Compare IDs and preprocessing before fitting: ' + str(y.value_counts().to_dict()))
    return X.copy(), y.copy()



## 3. WFCM implementation


In [ ]:
"""WFCM fitting and fixed-(k,m) parametric bootstrap from the manuscript.

Dependencies: numpy, scipy, scikit-learn. No data preprocessing is performed.
All likelihoods use f(x) proportional to exp(-h(x)/sigma**2), where
h(x) = [sum_j (w_j ||x-v_j||**2)**(-1/(m-1))]**(-(m-1)).
"""
import numpy as np
from scipy.optimize import minimize, linear_sum_assignment
from scipy.special import logsumexp
from scipy.spatial.distance import cdist
from scipy.stats import chi2, norm
from sklearn.mixture import GaussianMixture


def _data(X):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or min(X.shape) < 1 or not np.isfinite(X).all():
        raise ValueError('Expected a finite, nonempty samples-by-features matrix.')
    return X


import numpy as np
from scipy.special import logsumexp
from scipy.stats import qmc, norm, chi2, multivariate_t
from sklearn.mixture import GaussianMixture

class DefensiveProposal:
    """Data GMM + isotropic coverage at its centers + heavy-tailed defense.

    The isotropic component prevents thin data-covariance directions from
    undersampling the model's isotropic Gaussian tails. All densities are
    evaluated as the complete mixture, including the Student-t component.
    """
    def __init__(self, X, n_components, seed):
        self.seed = int(seed)
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full',
                                   random_state=self.seed).fit(X)
        if not self.gmm.converged_:
            raise RuntimeError('Importance-proposal GMM did not converge.')
        self.loc = X.mean(axis=0)
        self.d = X.shape[1]
        self.variance = max(float(np.var(X,axis=0,ddof=1).mean()), 1e-8)
        self.shape = 4.*self.variance*np.eye(self.d)
        self.df = 5.
        J = len(self.gmm.weights_)
        self.weights = np.r_[.2*self.gmm.weights_, np.full(J,.7/J), .1]
        self.name = '0.2 data GMM + 0.7 isotropic center mixture + 0.1 Student-t(df=5); scrambled Sobol'
    def sample(self, n):
        if int(n) != n or n < 2:
            raise ValueError('At least two integer integration samples are required.')
        n = int(n)
        U = qmc.Sobol(d=self.d+2, scramble=True, seed=self.seed).random_base2(int(np.ceil(np.log2(n))))[:n]
        U = np.clip(U, np.finfo(float).eps, 1-np.finfo(float).eps)
        labels = np.searchsorted(np.cumsum(self.weights), U[:,0], side='right')
        G = norm.ppf(U[:,1:self.d+1]);Z = np.empty((n,self.d));J=len(self.gmm.weights_)
        for j in range(J):
            use = labels == j
            Z[use] = self.gmm.means_[j] + G[use] @ np.linalg.cholesky(self.gmm.covariances_[j]).T
            use = labels == J+j
            Z[use] = self.gmm.means_[j] + G[use]*np.sqrt(self.variance)
        use = labels == 2*J
        radial = np.sqrt(self.df / chi2.ppf(U[use,-1], self.df))
        Z[use] = self.loc + G[use]*(2*np.sqrt(self.variance))*radial[:,None]
        return Z, labels
    def score_samples(self, Z):
        d2=((Z[:,None,:]-self.gmm.means_[None,:,:])**2).sum(axis=2)
        log_iso=logsumexp(-.5*d2/self.variance,axis=1)-np.log(len(self.gmm.weights_))-.5*self.d*np.log(2*np.pi*self.variance)
        log_t=multivariate_t.logpdf(Z,loc=self.loc,shape=self.shape,df=self.df)
        return logsumexp(np.vstack([np.log(.2)+self.gmm.score_samples(Z),
                                   np.log(.7)+log_iso,np.log(.1)+log_t]),axis=0)

def build_proposal(X, num_samples=32768, n_components=None, seed=42):
    X = np.asarray(X,dtype=float)
    if X.ndim != 2 or X.shape[0]<2 or X.shape[1]<1 or not np.isfinite(X).all():
        raise ValueError('Proposal requires finite data with at least two rows and one feature.')
    components = min(4,len(X)) if n_components is None else n_components
    proposal = DefensiveProposal(X,components,seed)
    Z,_ = proposal.sample(num_samples)
    return Z,proposal.score_samples(Z),proposal


def _unpack(theta, k, d):
    centers = theta[1:1+k*d].reshape(k, d)
    logits = np.r_[theta[1+k*d:], 0.0]
    return theta[0], centers, logits-logsumexp(logits)


def _pack(sigma, centers, w):
    w = np.asarray(w, dtype=float)
    if not np.isfinite(sigma) or sigma <= 0 or np.any(w <= 0) or not np.isfinite(w).all():
        raise ValueError('The density requires positive scale and strictly positive weights.')
    w = w/w.sum()
    return np.r_[np.log(sigma), np.asarray(centers).ravel(), np.log(w[:-1])-np.log(w[-1])]


def _terms(X, centers, logw, m):
    # Use log space; only distances at floating-point zero receive a floor.
    d2 = np.maximum(cdist(X, centers, 'sqeuclidean'), 1e-200)
    scores = -(np.log(d2)+logw)/(m-1)
    lse = logsumexp(scores, axis=1)
    return np.exp(scores-lse[:, None]), np.exp(-(m-1)*lse), d2


def memberships(X, centers, w, m):
    if m <= 1:
        raise ValueError('m must exceed 1.')
    return _terms(_data(X), np.asarray(centers), np.log(w), m)[0]


def nll_gradient(theta, X, Z, logq, m, k, compute_center_gradient=True):
    """IS NLL and analytic gradient in log-scale / center / weight-logit coordinates."""
    logsigma, centers, logw = _unpack(theta, k, X.shape[1])
    w = np.exp(logw)
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        ux, hx, dx = _terms(X, centers, logw, m)
        uz, hz, dz = _terms(Z, centers, logw, m)
        ex, ez = hx*np.exp(-2*logsigma), hz*np.exp(-2*logsigma)
        lr = -ez-logq
        lse = logsumexp(lr)
        prob = np.exp(lr-lse)
        value = len(X)*(lse-np.log(len(Z))) + ex.sum()

        def gradient(A, U, e, d2, a):
            t = a[:, None]*e[:, None]*U/d2
            # Accumulate displacements directly: subtracting two large matrix
            # products loses precision when a center nearly equals a data point.
            if compute_center_gradient:
                displacement = centers[None, :, :]-A[:, None, :]
                gv = 2*np.einsum('ij,ijd->jd', t, displacement)
            else:
                gv = np.zeros_like(centers)
            gw = ((a*e)[:, None]*(U-w)).sum(axis=0)[:-1]
            return np.r_[-2*np.dot(a, e), gv.ravel(), gw]

        grad = gradient(X, ux, ex, dx, np.ones(len(X)))-gradient(Z, uz, ez, dz, len(X)*prob)
    if not np.isfinite(value) or not np.isfinite(grad).all():
        return np.inf, np.zeros_like(theta)
    return float(value), grad


def importance_diagnostics(fit, Z, logq):
    _, h, _ = _terms(Z, fit['centers'], np.log(fit['w']), fit['m'])
    lr = -h/fit['sigma']**2-logq
    lse = logsumexp(lr)
    p = np.exp(lr-lse)
    return dict(logZ=float(lse-np.log(len(Z))), ESS=float(1/np.sum(p*p)),
                ESS_fraction=float(1/(len(Z)*np.sum(p*p))), largest_IS_weight=float(p.max()))


def fit_fixed(X, Z, logq, m, k, initial_sigma=1., initial_centers=None,
              initial_w=None, seed=321, max_outer=200, epsilon=1e-5,
              delta=1e-5, inner_maxiter=1000, K_post=1000):
    """Algorithm emfuzzy for one m: block updates, then joint refinement.

    max_outer is an explicit operational guard; hitting it is reported as
    nonconvergence, never silently treated as satisfying the stopping rule.
    The weight softmax enforces the simplex throughout optimization.
    """
    X, Z = _data(X), _data(Z)
    logq = np.asarray(logq, dtype=float)
    if (m <= 1 or not np.isfinite(m) or not 1 <= k <= len(X)
            or X.shape[1] != Z.shape[1] or logq.shape != (len(Z),)
            or not np.isfinite(logq).all() or max_outer < 1 or inner_maxiter < 1
            or K_post < 0 or epsilon < 0 or delta < 0):
        raise ValueError('Invalid model, proposal, or optimizer configuration.')
    d = X.shape[1]
    centers = (X[np.random.default_rng(seed).choice(len(X), k, replace=False)].copy()
               if initial_centers is None else np.array(initial_centers, dtype=float, copy=True))
    if centers.shape != (k, d):
        raise ValueError('Initial centers must have shape (k,d).')
    w = np.full(k, 1/k) if initial_w is None else np.asarray(initial_w, dtype=float)
    if w.shape != (k,):
        raise ValueError('Initial weights must have shape (k,).')
    theta = _pack(initial_sigma, centers, w)
    value, _ = nll_gradient(theta, X, Z, logq, m, k)
    if not np.isfinite(value):
        raise RuntimeError('Initial likelihood is not finite.')
    sw_indices = np.r_[0, np.arange(1+k*d, len(theta))]
    history, converged, failures = [], False, 0

    def physical(t):
        ls, v, lw = _unpack(t, k, d)
        return np.r_[np.exp(ls), v.ravel(), np.exp(lw)]

    for iteration in range(1, max_outer+1):
        old_theta, old_value = theta.copy(), value
        _, v, lw = _unpack(theta, k, d)
        U, _, _ = _terms(X, v, lw, m)
        # The centroid update is unconditional, exactly as in the manuscript.
        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        um = np.exp(logum-logsumexp(logum, axis=0))
        theta[1:1+k*d] = (um.T@X).ravel()

        def block_objective(sw):
            candidate = theta.copy()
            candidate[sw_indices] = sw
            f, g = nll_gradient(candidate, X, Z, logq, m, k, compute_center_gradient=False)
            return f, g[sw_indices]

        opt = minimize(block_objective, theta[sw_indices], jac=True, method='L-BFGS-B',
                       options=dict(maxiter=inner_maxiter, ftol=1e-12, gtol=1e-6, maxls=100))
        if not opt.success:
            failures += 1
        if not np.isfinite(opt.fun):
            raise RuntimeError('Nonfinite scale/weight optimization.')
        theta[sw_indices] = opt.x
        value, _ = nll_gradient(theta, X, Z, logq, m, k)
        step = float(np.linalg.norm(physical(theta)-physical(old_theta)))
        change = abs(value-old_value)
        history.append(dict(stage='blockwise', iteration=iteration, NLL=value,
                            parameter_change=step, NLL_change=change,
                            optimizer_success=bool(opt.success), message=str(opt.message)))
        converged = bool(opt.success and (step <= epsilon or change <= delta))
        if converged:
            break

    post_success, post_message, post_iterations, post_status = True, 'K_post=0: refinement skipped', 0, 0
    post_attempts = 0
    # A fresh L-BFGS history can recover from an abnormal line search. Both
    # attempts share the manuscript's TOTAL K_post iteration budget.
    if K_post:
        post_success = False
        for attempt, line_search_budget in enumerate((100, 200), start=1):
            remaining = K_post-post_iterations
            if remaining <= 0:
                break
            opt = minimize(nll_gradient, theta, args=(X, Z, logq, m, k), jac=True,
                           method='L-BFGS-B', options=dict(maxiter=remaining,
                           ftol=1e-12, gtol=1e-6, maxls=line_search_budget))
            trial_value, trial_gradient = nll_gradient(opt.x, X, Z, logq, m, k)
            accepted = bool(np.isfinite(trial_value) and trial_value <= value+1e-7)
            if accepted:
                theta, value = opt.x.copy(), float(trial_value)
            post_iterations += int(opt.nit)
            post_attempts = attempt
            post_success = bool(opt.success and accepted)
            post_status = int(opt.status) if accepted else 2
            post_message = str(opt.message) if accepted else 'Refinement returned a nonfinite or worsening point.'
            history.append(dict(stage='joint_refinement', iteration=post_iterations,
                                attempt=attempt, attempt_iterations=int(opt.nit),
                                NLL=value, optimizer_success=post_success,
                                optimizer_status=post_status, message=post_message,
                                gradient_inf_norm=float(np.max(np.abs(trial_gradient))),
                                accepted=accepted, maxls=line_search_budget))
            if post_success or post_status == 1:
                break
    ls, centers, lw = _unpack(theta, k, d)
    result = dict(sigma=float(np.exp(ls)), centers=centers.copy(), w=np.exp(lw),
                  U=_terms(X, centers, lw, m)[0], m=float(m), k=int(k), NLL=float(value),
                  theta=theta.copy(), history=history, converged=converged,
                  optimizer_failures=failures, post_success=post_success,
                  post_message=post_message, post_iterations=post_iterations,
                  post_status=post_status, post_attempts=post_attempts,
                  seed=int(seed), n_iter=iteration, engine='manuscript_defensive_rqmc',
                  fit_options=dict(max_outer=max_outer, epsilon=epsilon, delta=delta,
                                   inner_maxiter=inner_maxiter, K_post=K_post))
    result['IS_diagnostics'] = importance_diagnostics(result, Z, logq)
    return result


def select_m(X, k, m_grid, num_is=32768, proposal_components=None,
             proposal_seed=674, seeds=(321,), **fit_options):
    """Fixed k; fit each m, refine, and select the smallest NLL (no XB filter)."""
    X = _data(X)
    grid = np.asarray(m_grid, dtype=float)
    if grid.ndim != 1 or not len(grid) or not np.isfinite(grid).all() or np.any(grid <= 1):
        raise ValueError('m_grid must contain finite values greater than one.')
    if not len(seeds):
        raise ValueError('Provide at least one initialization seed.')
    Z, logq, proposal = build_proposal(X, num_is, proposal_components, proposal_seed)
    fits, rows = [], []
    for m in grid:
        candidates = [fit_fixed(X, Z, logq, float(m), k, seed=int(s), **fit_options) for s in seeds]
        fit = min(candidates, key=lambda f: f['NLL'])
        fits.append(fit)
        rows.append(dict(m=float(m), NLL=fit['NLL'], converged=fit['converged'],
                         optimizer_failures=fit['optimizer_failures'],
                         post_success=fit['post_success'], post_status=fit['post_status'],
                         post_message=fit['post_message'], post_iterations=fit['post_iterations'],
                         post_attempts=fit['post_attempts'], seed=fit['seed']))
    best = min(fits, key=lambda f: f['NLL'])
    return best, rows, dict(samples=Z, logpdf=logq, model=proposal), fits


def sample_iid(fit, n, seed=123, max_proposals=10_000_000):
    """Independent rejection draws, with a global analytic Gaussian envelope.

    Let a_j = w_j ||x-v_j||^2 and C=k**(m-1). Then
    h(x) >= min_j(a_j)/C, hence exp(-h/sigma^2) <=
    sum_j exp(-w_j ||x-v_j||^2/(sigma^2*C)). Each summand is an
    unnormalized isotropic Gaussian. Select its component proportional to
    its integral and accept with exp(-h/sigma^2)/sum_j exp(-a_j/(sigma^2*C)).
    No fitted normalizing constant, burn-in, or MCMC approximation is needed.
    """
    if n < 1 or int(n) != n:
        raise ValueError('n must be a positive integer.')
    n = int(n)
    rng = np.random.default_rng(seed)
    centers, w = np.asarray(fit['centers']), np.asarray(fit['w'])
    sigma, m = fit['sigma'], fit['m']
    _pack(sigma, centers, w)
    if m <= 1 or not np.isclose(w.sum(), 1):
        raise ValueError('Invalid fitted model.')
    k, d = centers.shape
    logvar = 2*np.log(sigma)+(m-1)*np.log(k)-np.log(w)-np.log(2.)
    logmass = .5*d*logvar
    component_prob = np.exp(logmass-logsumexp(logmass))
    sd = np.exp(.5*logvar)
    chunks, accepted, proposed = [], 0, 0
    while accepted < n:
        count = min(max(256, 2*(n-accepted)), 8192, max_proposals-proposed)
        if count <= 0:
            raise RuntimeError('Rejection sampler proposal budget exceeded; no shortened dataset is returned.')
        labels = rng.choice(k, size=count, p=component_prob)
        Y = centers[labels]+rng.normal(size=(count, d))*sd[labels, None]
        _, h, _ = _terms(Y, centers, np.log(w), m)
        d2 = cdist(Y, centers, 'sqeuclidean')
        log_envelope = logsumexp(-d2/(2*np.exp(logvar)), axis=1)
        log_accept = -h/sigma**2-log_envelope
        if np.max(log_accept) > 1e-8:
            raise RuntimeError('Numerical violation of rejection envelope.')
        keep = Y[np.log(rng.random(count)) <= np.minimum(log_accept, 0)]
        chunks.append(keep)
        accepted += len(keep)
        proposed += count
    return np.concatenate(chunks)[:n]


def align_parameters(centers, w, observed_centers):
    rows, cols = linear_sum_assignment(cdist(observed_centers, centers, 'sqeuclidean'))
    order = np.empty(len(rows), dtype=int)
    order[rows] = cols
    return np.asarray(centers)[order].copy(), np.asarray(w)[order].copy(), order


def bootstrap_summary(estimates, k, d, alpha=.05):
    A = np.asarray(estimates, dtype=float)
    if (A.ndim != 2 or A.shape[1] != 1+(d+1)*k or len(A) < max(d+1, k)
            or not np.isfinite(A).all() or not 0 < alpha < 1 or k < 2):
        raise ValueError('Invalid bootstrap array, B, alpha, or k (requires k >= 2).')
    B = len(A)
    mean, sd = A.mean(axis=0), A.std(axis=0, ddof=0)
    z = norm.ppf(1-alpha/2)
    ci = np.column_stack([mean-z*sd, mean+z*sd])
    ci[0, 0] = max(0., ci[0, 0])  # endpoint 0 denotes intersection with sigma > 0
    ci[1+k*d:, :] = np.clip(ci[1+k*d:, :], 0., 1.)
    center_draws = A[:, 1:1+k*d].reshape(B, k, d)
    covariance = np.stack([np.atleast_2d(np.cov(center_draws[:, j, :], rowvar=False, ddof=1)) for j in range(k)])
    # The manuscript requires an inverse, not a center-covariance pseudoinverse.
    if any(np.linalg.matrix_rank(C) < d for C in covariance):
        raise RuntimeError('Singular center covariance: increase B/check failed or degenerate refits.')
    weight_cov = np.atleast_2d(np.cov(A[:, 1+k*d:], rowvar=False, ddof=1))
    if np.linalg.matrix_rank(weight_cov) < k-1:
        raise RuntimeError('Weight covariance has rank below k-1; increase B/check degeneracy.')
    return dict(B=B, k=k, d=d, alpha=alpha, mean=mean, sd=sd, ci=ci,
                scale_lower_open=bool(ci[0, 0] == 0),
                center_means=mean[1:1+k*d].reshape(k, d),
                center_covariance=covariance, center_precision=np.linalg.inv(covariance),
                center_cutoff=float(chi2.ppf(1-alpha, d)),
                weight_mean=mean[1+k*d:], weight_covariance=weight_cov,
                weight_precision=np.linalg.pinv(weight_cov, hermitian=True),
                weight_cutoff=float(chi2.ppf(1-alpha, k-1)))


def in_center_region(points, summary, center):
    delta = np.asarray(points)-summary['center_means'][center]
    return np.einsum('...i,ij,...j->...', delta, summary['center_precision'][center], delta) <= summary['center_cutoff']


def in_weight_region(weights, summary, atol=1e-10):
    W = np.asarray(weights, dtype=float)
    delta = W-summary['weight_mean']
    quadratic = np.einsum('...i,ij,...j->...', delta, summary['weight_precision'], delta)
    simplex = (W >= 0).all(axis=-1) & (W <= 1).all(axis=-1) & np.isclose(W.sum(axis=-1), 1., atol=atol, rtol=0.)
    return simplex & (quadratic <= summary['weight_cutoff'])


def parametric_bootstrap(X, fit, B=100, alpha=.05, num_is=32768,
                         proposal_components=None, seed=999, fit_options=None):
    """Algorithm inference: iid data, rebuilt proposal/cloud, fixed k,m, alignment."""
    X = _data(X)
    k, d = fit['centers'].shape
    if k < 2 or B < max(d+1, k) or int(B) != B or not 0 < alpha < 1:
        raise ValueError('Require k >= 2, integer B >= max(d+1,k), and 0 < alpha < 1.')
    if (X.shape[1] != d or not fit['converged'] or fit['optimizer_failures']
            or (not fit['post_success'] and fit['post_status'] != 1)):
        raise RuntimeError('Check dimensions and observed blockwise convergence before bootstrapping.')
    options = dict(fit.get('fit_options', {}))
    if fit_options:
        options.update(fit_options)
    rng = np.random.default_rng(seed)
    estimates, diagnostics = [], []
    for b in range(int(B)):
        sample_seed, proposal_seed, refit_seed = map(int, rng.integers(0, 2**31-1, size=3))
        Y = sample_iid(fit, len(X), seed=sample_seed)
        Z, logq, _ = build_proposal(Y, num_is, proposal_components, proposal_seed)
        refit = fit_fixed(Y, Z, logq, fit['m'], k, initial_sigma=fit['sigma'],
                          initial_centers=fit['centers'], initial_w=fit['w'],
                          seed=refit_seed, **options)
        if (not refit['converged'] or refit['optimizer_failures']
                or (not refit['post_success'] and refit['post_status'] != 1)):
            raise RuntimeError(f'Bootstrap replicate {b+1} did not converge cleanly; no partial inference returned.')
        centers, w, order = align_parameters(refit['centers'], refit['w'], fit['centers'])
        estimates.append(np.r_[refit['sigma'], centers.ravel(), w])
        diagnostics.append(dict(replicate=b+1, sample_seed=sample_seed,
                                proposal_seed=proposal_seed, refit_seed=refit_seed,
                                permutation=order.tolist(), NLL=refit['NLL'],
                                post_success=refit['post_success'],
                                IS_diagnostics=refit['IS_diagnostics']))
        print(f'Bootstrap {b+1}/{B}: fixed k={k}, m={fit["m"]:.6g}', flush=True)
    estimates = np.asarray(estimates)
    return dict(estimates=estimates, summary=bootstrap_summary(estimates, k, d, alpha), diagnostics=diagnostics)


from pathlib import Path
import json
import pandas as pd

def report_inference(bootstrap, output_dir):
    summary = bootstrap['summary']
    k, d = summary['k'], summary['d']
    names = ['sigma']+[f'v{a+1}_{j+1}' for a in range(k) for j in range(d)]+[f'w{a+1}' for a in range(k)]
    table = pd.DataFrame({'parameter': names, 'bootstrap_mean': summary['mean'],
                          'bootstrap_sd_ddof0': summary['sd'],
                          'lower': summary['ci'][:, 0], 'upper': summary['ci'][:, 1]})
    display(table)
    if summary['scale_lower_open']:
        print('The scale interval lower endpoint 0 is excluded (sigma > 0).')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir/'bootstrap_scalar_intervals.csv', index=False)
    np.savez_compressed(output_dir/'bootstrap_regions.npz', estimates=bootstrap['estimates'],
                        **{key: value for key, value in summary.items()})
    (output_dir/'bootstrap_diagnostics.json').write_text(json.dumps(bootstrap['diagnostics'], indent=2))
    print('Regions saved to', output_dir/'bootstrap_regions.npz')
    print('Use in_center_region(points, summary, center_index) and in_weight_region(weights, summary).')
    return summary, table

# Compatibility for the separate exploratory LR cell; it does not select m.
def em_like_fuzzy_clustering_c(data, proposal_samples, proposal_weights, m, c,
                                initial_sigma=1., initial_centers=None, initial_w=None,
                                max_iter=200, tol=1e-5, random_state=321):
    if np.any(np.asarray(proposal_weights) <= 0):
        raise ValueError('Nonpositive proposal pdf; use logpdf with fit_fixed instead.')
    return fit_fixed(data, proposal_samples, np.log(proposal_weights), m, c,
                     initial_sigma=initial_sigma, initial_centers=initial_centers,
                     initial_w=initial_w, max_outer=max_iter, delta=tol, seed=random_state)

def partial_negative_log_likelihood_c(params, data, U, proposal_samples, proposal_weights, m):
    # U is retained only for compatibility; this is the actual model likelihood.
    d = data.shape[1]
    k = (len(params)-1)//(d+1)
    params = np.asarray(params)
    theta = _pack(params[0], params[1:1+k*d].reshape(k,d), params[1+k*d:])
    return nll_gradient(theta, data, proposal_samples, np.log(proposal_weights), m, k)[0]

from scipy.spatial.distance import pdist
def weighted_xie_beni(X, U, centers, m, w):
    """Cell 5: sum_ij K*w_j*U_ij^m*d_ij^2 / (N*min center distance^2)."""
    sep = pdist(centers, metric='sqeuclidean').min()
    if not np.isfinite(sep) or sep < 1e-14:
        return np.inf
    return float(np.sum(len(w) * w * U**m * cdist(X, centers, 'sqeuclidean'))
                 / (len(X) * sep))

def bounded_normalizer_audit(fit,X,N=1000000,seed=20261104,tolerance=2.,batch_size=10000):
    # Independent diagnostic only: never modifies parameters or chooses m.
    V=np.asarray(fit['centers']);weights=np.asarray(fit['w']);sigma=fit['sigma'];m=fit['m'];k,d=V.shape
    rng=np.random.default_rng(seed)
    lv=2*np.log(sigma)+(m-1)*np.log(k)-np.log(weights)-np.log(2.)
    lmass=.5*d*lv;component=np.exp(lmass-logsumexp(lmass));logG=logsumexp(lmass)+.5*d*np.log(2*np.pi)
    total=total2=0.
    for start in range(0,N,batch_size):
        count=min(batch_size,N-start);z=rng.choice(k,size=count,p=component)
        Y=V[z]+rng.normal(size=(count,d))*np.exp(.5*lv)[z,None]
        _,h,_=_terms(Y,V,np.log(weights),m)
        logenv=logsumexp(-cdist(Y,V,'sqeuclidean')/(2*np.exp(lv)),axis=1)
        logratio=-h/sigma**2-logenv
        if np.max(logratio)>1e-8:raise RuntimeError('Independent audit envelope violated.')
        ratios=np.exp(np.minimum(logratio,0));total+=ratios.sum();total2+=np.dot(ratios,ratios)
    mean=total/N
    if mean<=0:raise RuntimeError('Independent normalizer audit has no accepted mass.')
    relse=np.sqrt(max(0,total2/N-mean**2)/N)/mean
    _,hx,_=_terms(np.asarray(X),V,np.log(weights),m);logZ=float(logG+np.log(mean));nll=float(len(X)*logZ+(hx/sigma**2).sum())
    result=dict(method='bounded analytic Gaussian-envelope integration',samples=N,seed=seed,logZ=logZ,NLL=nll,estimated_logZ_standard_error=float(relse),estimated_NLL_standard_error=float(len(X)*relse),mean_acceptance=float(mean),purpose='independent diagnostic only; not used for fitting or selection',fitting_cloud_NLL=fit['NLL'])
    result['NLL_difference_from_fitting_cloud']=nll-fit['NLL'];result['diagnostic_threshold']=max(tolerance,5*result['estimated_NLL_standard_error'])
    result['status']='normalizer_check_passed' if abs(result['NLL_difference_from_fitting_cloud'])<=result['diagnostic_threshold'] else 'requires_normalizer_accuracy_review'
    return result


## 4. Ordinary FCM comparison
Ordinary FCM fits the centers by fuzzy distortion. For each candidate m, it keeps those centers and equal weights fixed, profiles the scale in the equal-weight WFCM density, and selects m by minimum numerical NLL. WFCM jointly refines its own parameters.


In [ ]:
def fit_fcm(X, m, K=4, seed=123, maxiter=1000, error=1e-6):
    # Standard fuzzy c-means, random membership initialization as in prior code.
    if m <= 1:
        raise ValueError('m must be greater than one.')
    rng = np.random.RandomState(seed)
    U = rng.rand(K, len(X)).T
    U /= U.sum(axis=1, keepdims=True)
    logw = np.full(K, -np.log(K))
    converged = False
    for it in range(1, maxiter+1):
        logum = m*np.log(np.maximum(U, np.finfo(float).tiny))
        weights = np.exp(logum-logsumexp(logum, axis=0))
        centers = weights.T @ X
        updated, _, _ = _terms(X, centers, logw, m)
        delta = np.linalg.norm(updated-U)
        U = updated
        if delta < error:
            converged = True
            break
    objective = float(np.sum(U**m * cdist(X, centers, 'sqeuclidean')))
    return dict(U=U, centers=centers, w=np.ones(K)/K, m=float(m), seed=seed,
                objective=objective, converged=converged, n_iter=it)

def select_fcm(X, grid, seeds, num_is):
    """Select m by the equal-weight likelihood at the ordinary FCM centers."""
    from scipy.optimize import brentq

    Z, logq, _ = build_proposal(X, num_is, PROPOSAL_COMPONENTS, IS_SEED)
    logw = np.full(K, -np.log(K))
    rows, fits = [], []
    for m in grid:
        candidates = [fit_fcm(X, m, K, seed) for seed in seeds]
        converged = [fit for fit in candidates if fit['converged']]
        fit = min(converged or candidates, key=lambda fit: fit['objective'])
        fits.append(fit)
        row = dict(m=m, seed=fit['seed'], objective=fit['objective'],
                   converged=fit['converged'], n_iter=fit['n_iter'],
                   sigma=np.nan, NLL=np.nan)
        rows.append(row)
        if not fit['converged']:
            continue

        # Profile precision with centers and equal weights held fixed.
        mean_h = _terms(X, fit['centers'], logw, m)[1].mean()
        if not np.isfinite(mean_h) or mean_h <= 0:
            continue
        h = _terms(Z, fit['centers'], logw, m)[1] / mean_h

        def derivative(t):
            log_weights = -t * h - logq
            return 1 - np.dot(np.exp(log_weights - logsumexp(log_weights)), h)

        # The numerical NLL is convex in t = mean_h / sigma**2.
        upper = 1.
        while derivative(upper) <= 0 and upper < 2**32:
            upper *= 2
        if derivative(0.) >= 0 or derivative(upper) <= 0:
            continue
        t = brentq(derivative, 0., upper)
        sigma = np.sqrt(mean_h / t)
        nll = len(X) * (t + logsumexp(-t * h - logq) - np.log(len(Z)))
        row.update(sigma=sigma, NLL=nll)
        fit.update(sigma=sigma, NLL=nll)

    table = pd.DataFrame(rows)
    valid = np.isfinite(table.NLL)
    if not valid.any():
        raise RuntimeError('No converged FCM candidate has a finite interior likelihood optimum.')
    return fits[table.loc[valid, 'NLL'].idxmin()], table


## 5. One evaluation function for all three methods

In [ ]:
def evaluate_method(fit, labels, method):
    U = fit['U']
    if U.shape != (len(labels), K) or not np.isfinite(U).all() or np.any(U < 0) or not np.allclose(U.sum(axis=1), 1):
        raise ValueError('Invalid membership matrix.')
    y = labels.to_numpy()
    hard = U.argmax(axis=1)
    contingency = np.array([[np.sum((y == s) & (hard == k)) for k in range(K)] for s in SUBTYPES])
    a, b = linear_sum_assignment(-contingency)
    mapping = {int(j): SUBTYPES[i] for i, j in zip(a, b)}
    colorder = [next(k for k in range(K) if mapping[k] == s) for s in SUBTYPES]
    cm = pd.DataFrame(contingency[:, colorder], index=pd.Index(SUBTYPES, name='PAM50'),
                      columns=pd.Index(SUBTYPES, name=method))
    recall = pd.Series(np.diag(cm)/cm.sum(axis=1), index=cm.index, name=method)
    members = pd.DataFrame(U[:, colorder], index=labels.index, columns=SUBTYPES)
    members['PAM50'] = y
    members['cluster_id'] = hard
    members['mapped_cluster'] = [mapping[j] for j in hard]
    members['max_membership'] = U.max(axis=1)
    members['uncertainty'] = 1-U.max(axis=1)
    members['entropy_nats'] = -xlogy(U, U).sum(axis=1)
    members['entropy_normalized'] = members.entropy_nats/np.log(K)
    order = np.argsort(-members[SUBTYPES].to_numpy(), axis=1, kind='stable')
    members['top1_cluster'] = np.array(SUBTYPES)[order[:, 0]]
    members['top2_cluster'] = np.array(SUBTYPES)[order[:, 1]]
    sorted_u = np.sort(U, axis=1)
    members['top_gap'] = sorted_u[:, -1]-sorted_u[:, -2]
    members['LumA_LumB_gap'] = abs(members.LumA-members.LumB)
    boundary = members.loc[members.PAM50.isin(['LumA','LumB']) &
                           members.top1_cluster.isin(['LumA','LumB']) &
                           members.top2_cluster.isin(['LumA','LumB'])].sort_values('top_gap')
    sizes = pd.DataFrame({'mapped_subtype': [mapping[j] for j in range(K)],
                          'hard_n': np.bincount(hard, minlength=K),
                          'soft_mass': U.sum(axis=0),
                          'estimated_distance_weight': fit.get('w', np.full(K, np.nan))},
                         index=pd.Index(range(K), name='cluster_id'))
    if method != 'WFCM':
        sizes['estimated_distance_weight'] = np.nan
    if (sizes.hard_n == 0).any():
        warnings.warn(f'{method}: empty hard cluster; its subtype name is arbitrary.')
    summary = {'method': method, 'm': fit.get('m', np.nan),
               'ARI': adjusted_rand_score(y, hard), 'NMI': normalized_mutual_info_score(y, hard),
               'hard_clusters': int(np.count_nonzero(sizes.hard_n)),
               'mean_max_membership': members.max_membership.mean(),
               'mean_entropy_normalized': members.entropy_normalized.mean(),
               'converged': fit.get('converged', True)}
    summary.update({s+'_recall': recall[s] for s in SUBTYPES})
    return dict(summary=summary, mapping=mapping, confusion_matrix=cm,
                confusion_row_normalized=cm.div(cm.sum(axis=1), axis=0), subtype_recall=recall,
                membership_df=members, raw_memberships=pd.DataFrame(U, index=labels.index, columns=[f'C{k}' for k in range(K)]),
                soft_confusion=members.groupby('PAM50')[SUBTYPES].mean().reindex(SUBTYPES),
                uncertainty_summary=members.groupby('PAM50')[['max_membership','uncertainty','entropy_nats','entropy_normalized']].agg(['mean','median']).reindex(SUBTYPES),
                cluster_sizes=sizes, luminal_boundary=boundary)


def plot_comparison(evaluations, X, labels):
    methods = list(evaluations)
    fig, axes = plt.subplots(2, len(methods), figsize=(14, 8), constrained_layout=True)
    for j, method in enumerate(methods):
        for row, key, title in [(0, 'confusion_row_normalized', 'Hard recall by subtype'),
                                 (1, 'soft_confusion', 'Mean membership')]:
            values = evaluations[method][key].to_numpy()
            ax = axes[row, j]
            ax.imshow(values, cmap='Blues', vmin=0, vmax=1)
            ax.set(xticks=range(K), yticks=range(K), xticklabels=SUBTYPES, yticklabels=SUBTYPES,
                   title=method+' — '+title, xlabel='Mapped cluster', ylabel='PAM50')
            for a in range(K):
                for b in range(K):
                    ax.text(b,a,f'{values[a,b]:.3f}',ha='center',va='center',color='white' if values[a,b]>.5 else 'black')
    fig2, axes2 = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    groups = [('PAM50', labels.to_numpy())] + [(m, evaluations[m]['membership_df'].mapped_cluster.to_numpy()) for m in methods]
    for ax, (name, group) in zip(axes2, groups):
        for s, color in zip(SUBTYPES, ['#0072B2','#D55E00','#009E73','#CC79A7']):
            mask = group == s
            ax.scatter(X[mask,0],X[mask,1],s=10,alpha=.6,label=s,color=color)
        ax.set(title=name, xlabel='PC1', ylabel='PC2')
    axes2[-1].legend(fontsize=8)
    return fig, fig2



## Load the supplied data
Raw/prepared modes establish row alignment explicitly. Existing-kernel input mode is omitted so a fresh-kernel run has no hidden data objects.

In [ ]:
selected_genes, pca_model = [], None
if INPUT_MODE == 'raw':
    X_pca, labels, analysis_info, pca_model, selected_genes = load_raw_data(
        EXPR_PATH, CLINICAL_SOURCE, STANDARDIZE_GENES)
elif INPUT_MODE == 'prepared':
    prepared = pd.read_csv(PREPARED_PATH, index_col=0)
    pc_cols = [f'PC{i+1}' for i in range(N_PC)]
    X_pca, labels = prepared[pc_cols], prepared.PAM50.copy()
else:
    raise ValueError('INPUT_MODE must be raw or prepared.')
X_pca, labels = check_input(X_pca, labels)
input_hash = hashlib.sha256(X_pca.astype('<f8').tobytes() + '\n'.join(map(str, labels.index)).encode()).hexdigest()
print('Input:', X_pca.shape, 'SHA256:', input_hash)
print(labels.value_counts())
if pca_model is not None:
    print('First two PC variance fractions:', pca_model.explained_variance_ratio_[:2])


label_hash=hashlib.sha256('\n'.join(labels.astype(str)).encode()).hexdigest()
if INPUT_MODE=='prepared' and STRICT_REFERENCE_INPUT:
    if input_hash!=EXPECTED_PREPARED_HASH or label_hash!=EXPECTED_LABEL_HASH:
        raise ValueError('Prepared scores/IDs/labels differ from the recorded reference. Check provenance before fitting, or explicitly disable STRICT_REFERENCE_INPUT for a different analysis.')
    print('Prepared scores, IDs and labels match the embedded reference fingerprints.')
print('Output folder:', OUTPUT_DIR)

## 7. Fit methods before inspecting external performance
WFCM fixes k=4 and selects m by refined NLL over M_GRID. MODE="fixed" is available as a singleton m-grid.


In [ ]:
if MODE not in ('fixed', 'select'):
    raise ValueError('MODE must be fixed or select.')
grid = [FIXED_M] if MODE == 'fixed' else list(M_GRID)
kmeans = KMeans(n_clusters=K, n_init=100, random_state=123).fit(X_pca)
km_fit = {'U': np.eye(K)[kmeans.labels_], 'centers': kmeans.cluster_centers_,
          'converged': bool(kmeans.n_iter_ < kmeans.max_iter)}
fcm_fit, fcm_search = select_fcm(X_pca, grid, FCM_SEEDS, NUM_IS)
wfcm_fit, search_rows, importance, candidate_fits = select_m(
    X_pca, K, grid, num_is=NUM_IS, proposal_components=PROPOSAL_COMPONENTS,
    proposal_seed=IS_SEED, seeds=WFCM_SEEDS, max_outer=MAX_OUTER,
    inner_maxiter=INNER_MAXITER, K_post=K_POST)
wfcm_search = pd.DataFrame(search_rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wfcm_search.to_csv(OUTPUT_DIR/'wfcm_search.csv', index=False)
invalid_candidates = [f for f in candidate_fits if
    not f['converged'] or f['optimizer_failures'] or
    (not f['post_success'] and f['post_status'] != 1)]
if invalid_candidates:
    details = [f"m={f['m']:.6g}: blockwise_converged={f['converged']}, "
               f"iterations={f['n_iter']}/{f['fit_options']['max_outer']}, "
               f"blockwise_failures={f['optimizer_failures']}, "
               f"refinement_status={f['post_status']}, message={f['post_message']}"
               for f in invalid_candidates]
    raise RuntimeError('Unresolved candidate fitting problems:\n'+'\n'.join(details))
wfcm_search = pd.DataFrame(search_rows)
# Keep the fit dictionary unchanged for inference and reproducibility.
wfcm_history = pd.DataFrame(wfcm_fit['history'])
wfcm_is_diagnostics = pd.DataFrame([wfcm_fit['IS_diagnostics']], index=['fit_draws'])
fits = {'K-means': km_fit, 'FCM': fcm_fit, 'WFCM': wfcm_fit}
display(fcm_search)
display(wfcm_search)
display(wfcm_history)
display(wfcm_is_diagnostics)
print('WFCM selected m by NLL:', wfcm_fit['m'])

# Record IS quality for every selected-per-m candidate; no additional selection rule.
wfcm_candidate_diagnostics = pd.DataFrame([dict(m=f['m'], **f['IS_diagnostics']) for f in candidate_fits])
wfcm_candidate_diagnostics.to_csv(OUTPUT_DIR/'wfcm_candidate_importance_diagnostics.csv', index=False)
display(wfcm_candidate_diagnostics)
# Independent IS evaluation is descriptive and never changes fitted parameters or m.
validation_Z, validation_logq, _ = build_proposal(X_pca, VALIDATION_NUM_IS, PROPOSAL_COMPONENTS, VALIDATION_SEED)
validation_nll, _ = nll_gradient(wfcm_fit['theta'], X_pca, validation_Z,
                                validation_logq, wfcm_fit['m'], K)
validation_diagnostics = importance_diagnostics(wfcm_fit, validation_Z, validation_logq)
validation_table = pd.DataFrame([dict(cloud='fitting', NLL=wfcm_fit['NLL'],
                                      **wfcm_fit['IS_diagnostics']),
                                 dict(cloud='independent', NLL=validation_nll,
                                      **validation_diagnostics)])
validation_table.to_csv(OUTPUT_DIR/'wfcm_independent_is_check.csv', index=False)
display(validation_table)
# Independently evaluate all m candidates before using their fitted NLL ranking.
validation_rows = []
for candidate in candidate_fits:
    value, _ = nll_gradient(candidate['theta'], X_pca, validation_Z,
                            validation_logq, candidate['m'], K)
    validation_rows.append(dict(m=candidate['m'], fitting_NLL=candidate['NLL'],
                                independent_NLL=value,
                                difference=value-candidate['NLL']))
candidate_validation = pd.DataFrame(validation_rows)
candidate_validation.to_csv(OUTPUT_DIR/'candidate_normalizer_validation.csv', index=False)
display(candidate_validation)
if (candidate_validation['difference'].abs() > NORMALIZER_NLL_TOL).any():
    raise RuntimeError('Independent NLL accuracy check failed for an m candidate. '
                       'Increase NUM_IS (power of two) and refit; inference has not started.')
if float(candidate_validation.loc[candidate_validation.independent_NLL.idxmin(), 'm']) != wfcm_fit['m']:
    raise RuntimeError('The m ranking changed on an independent integration cloud. '
                       'Increase NUM_IS and refit before inference.')


## 8. Compare results and save all patient-level outputs

In [ ]:
evaluations = {name: evaluate_method(fit, labels, name) for name, fit in fits.items()}
comparison = pd.DataFrame([evaluations[name]['summary'] for name in fits]).set_index('method')
display(comparison.round(4))
for name, result in evaluations.items():
    print('\n', name, 'mapping:', result['mapping'])
    for key in ['confusion_matrix','soft_confusion','cluster_sizes','uncertainty_summary']:
        print(key)
        display(result[key].round(4))
fig_heatmap, fig_pca = plot_comparison(evaluations, X_pca, labels)
plt.show()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR/'comparison.csv')
fcm_search.to_csv(OUTPUT_DIR/'fcm_search.csv', index=False)
wfcm_search.to_csv(OUTPUT_DIR/'wfcm_search.csv', index=False)
wfcm_history.to_csv(OUTPUT_DIR/'wfcm_optimization.csv', index=False)
wfcm_is_diagnostics.to_csv(OUTPUT_DIR/'wfcm_importance_diagnostics.csv')
for name, result in evaluations.items():
    stem = name.lower().replace('-', '')
    for key, value in result.items():
        if isinstance(value, (pd.DataFrame, pd.Series)):
            value.to_csv(OUTPUT_DIR/f'{stem}_{key}.csv')
    np.savez_compressed(OUTPUT_DIR/f'{stem}_fit.npz', U=fits[name]['U'],
                        centers=fits[name]['centers'], w=fits[name].get('w', np.array([])),
                        m=fits[name].get('m', np.nan), sigma=fits[name].get('sigma', np.nan))
prepared = pd.DataFrame(X_pca, index=labels.index, columns=[f'PC{i+1}' for i in range(N_PC)])
prepared['PAM50'] = labels.to_numpy()
prepared.to_csv(OUTPUT_DIR/'prepared_pcs.csv')
fig_heatmap.savefig(OUTPUT_DIR/'comparison_heatmaps.png', dpi=180)
fig_pca.savefig(OUTPUT_DIR/'comparison_pca.png', dpi=180)
metadata = {'input_mode': INPUT_MODE, 'input_sha256': input_hash,
            'standardize_genes_if_raw': STANDARDIZE_GENES, 'selected_genes': selected_genes,
            'expression_path': str(EXPR_PATH), 'clinical_source': str(CLINICAL_SOURCE),
            'shape': list(X_pca.shape), 'mode': MODE, 'm_grid': list(map(float,grid)),
            'wfcm_engine': WFCM_ENGINE, 'wxb_min_if_selection': WXB_MIN,
             'fcm_seeds': FCM_SEEDS, 'fcm_selection': 'minimum equal-weight NLL profiled over scale at FCM centers',
            'wfcm_seeds': WFCM_SEEDS, 'selected_initialization_seed': int(wfcm_fit['seed']),
            'initialization_rng': 'numpy.default_rng', 'k': K, 'selected_m': float(wfcm_fit['m']),
            'NLL': float(wfcm_fit['NLL']), 'bootstrap_seed': BOOTSTRAP_SEED,
            'bootstrap_status': 'not_run', 'output_dir': str(OUTPUT_DIR), 'is_samples': NUM_IS,
            'is_seed': IS_SEED, 'validation_seed': VALIDATION_SEED,
            'max_outer': MAX_OUTER, 'inner_maxiter': INNER_MAXITER,
            'fit_options': wfcm_fit['fit_options'],
            'K_post': K_POST, 'B': B, 'alpha': ALPHA, 'wfcm_selection': 'minimum refined NLL',
            'input_prepared_path': str(PREPARED_PATH),
            'likelihood_normalizer': 'importance sampling', 'weight_domain': 'strictly positive simplex',
            'conditioning': 'fixed input cohort, prepared PCs, k and selected m',
            'proposal': importance['model'].name, 'proposal_components': PROPOSAL_COMPONENTS,
            'validation_is_samples': VALIDATION_NUM_IS, 'normalizer_NLL_tolerance': NORMALIZER_NLL_TOL,
            'python': platform.python_version(), 'numpy': np.__version__,
            'scipy': scipy.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__}
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(metadata, indent=2))
print('Saved comparison and patient-level results to', OUTPUT_DIR.resolve())

In [ ]:
# Fixed-(k,m) manuscript bootstrap for breast-cancer WFCM.
normalizer_audit=bounded_normalizer_audit(wfcm_fit,X_pca,N=1000000,seed=20261104,tolerance=NORMALIZER_NLL_TOL)
(OUTPUT_DIR/'bounded_normalizer_check.json').write_text(json.dumps(normalizer_audit,indent=2))
metadata=json.loads((OUTPUT_DIR/'run_settings.json').read_text())
metadata.update(numerical_status=normalizer_audit['status'],normalizer_audit=normalizer_audit)
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(metadata,indent=2))
if metadata.get('numerical_status') != 'normalizer_check_passed':
    raise RuntimeError('Independent bounded-normalizer audit failed; bootstrap not started.')
metadata['bootstrap_status'] = 'running'
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(metadata, indent=2))
bootstrap = parametric_bootstrap(X_pca, wfcm_fit, B=B, alpha=ALPHA,
                               num_is=NUM_IS, proposal_components=PROPOSAL_COMPONENTS, seed=BOOTSTRAP_SEED)
inference, interval_table = report_inference(bootstrap, OUTPUT_DIR)
metadata['bootstrap_status'] = 'complete'
(OUTPUT_DIR/'run_settings.json').write_text(json.dumps(metadata, indent=2))
# All centers are five-dimensional. This is their marginal PC1–PC2 display,
# using the chi-square_2 cutoff; full 5D region parameters are saved above.
from matplotlib.patches import Ellipse
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_pca[:, 0], X_pca[:, 1], s=8, alpha=.25, color='gray')
for j in range(K):
    mu = inference['center_means'][j, :2]
    C = inference['center_covariance'][j, :2, :2]
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    radii = np.sqrt(eigenvalues * chi2.ppf(1-ALPHA, 2))
    angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
    ax.add_patch(Ellipse(mu, width=2*radii[1], height=2*radii[0], angle=angle,
                         fill=False, edgecolor=f'C{j}', linewidth=2))
    ax.scatter(*mu, marker='X', color=f'C{j}', label=f'Center {j+1}')
ax.set(xlabel='PC1', ylabel='PC2', title='Marginal bootstrap center regions in PC1–PC2')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'bootstrap_center_regions_pc12.png', dpi=180)
plt.show()

## Supplementary embedding and membership diagnostics

In [ ]:
# ── Cell S1: Install / import extra dependencies ──────────────────────────────
import os, gc, warnings
 
# Force numba / OpenMP to single thread — the most reliable crash fix
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_DISABLE_JIT', '0')   # keep JIT but single-threaded
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
# ── import UMAP with crash guard ──────────────────────────────────────────────
UMAP_AVAILABLE = False
try:
    import umap as umap_module          # import the module, not the class
    UMAP_AVAILABLE = True
    print('umap-learn imported OK.')
except Exception as e:
    warnings.warn(f'UMAP import failed ({e}). UMAP panels will be skipped; '
                  'all other supplementary figures will still be produced.')
 
SUBTYPE_COLORS = {
    'Basal': '#0072B2',
    'Her2':  '#D55E00',
    'LumA':  '#009E73',
    'LumB':  '#CC79A7',
}
 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   # already defined in main notebook
 

In [ ]:
# ── Cell S2: Compute alternative embeddings ───────────────────────────────────
print('Computing t-SNE…', flush=True)
tsne = TSNE(n_components=2, perplexity=40, learning_rate='auto',
            init='pca', random_state=42, max_iter=1000)
Z_tsne = tsne.fit_transform(X_pca)
print(f't-SNE KL divergence: {tsne.kl_divergence_:.4f}')
 
if UMAP_AVAILABLE:
    print('Computing UMAP…', flush=True)
    reducer = umap_module.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                        metric='euclidean', random_state=42)
    Z_umap = reducer.fit_transform(X_pca)
    print('UMAP done.')
else:
    Z_umap = None
 
embedding_dict = {'PCA (PC1–PC2)': X_pca[:, :2], 't-SNE': Z_tsne}
if Z_umap is not None:
    embedding_dict['UMAP'] = Z_umap
 
pd.DataFrame(Z_tsne, index=labels.index,
             columns=['tSNE1', 'tSNE2']).assign(PAM50=labels.to_numpy()).to_csv(
    OUTPUT_DIR / 'tsne_coords.csv')
if Z_umap is not None:
    pd.DataFrame(Z_umap, index=labels.index,
                 columns=['UMAP1', 'UMAP2']).assign(PAM50=labels.to_numpy()).to_csv(
        OUTPUT_DIR / 'umap_coords.csv')
 
 

In [ ]:
n_emb = len(embedding_dict)
fig, axes = plt.subplots(2, n_emb, figsize=(5 * n_emb, 9),
                         constrained_layout=True)
fig.suptitle('Figure S1 — PAM50 subtypes across embeddings', fontsize=11)

y_arr = labels.to_numpy()
 
for col, (name, Z) in enumerate(embedding_dict.items()):
    ax_all = axes[0, col]
    for s in SUBTYPES:
        mask = y_arr == s
        ax_all.scatter(Z[mask, 0], Z[mask, 1], s=12, alpha=0.6,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_all.set(title=name, xlabel='Dim 1', ylabel='Dim 2')
    if col == n_emb - 1:
        ax_all.legend(fontsize=8, loc='upper right')
 
    ax_lum = axes[1, col]
    for s, alpha in [('LumA', 0.5), ('LumB', 0.6)]:
        mask = y_arr == s
        ax_lum.scatter(Z[mask, 0], Z[mask, 1], s=14, alpha=alpha,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_lum.set(title=f'{name} — LumA vs LumB only',
               xlabel='Dim 1', ylabel='Dim 2')
    if col == n_emb - 1:
        ax_lum.legend(fontsize=8)
 
axes[0, 0].set_ylabel('All subtypes\n\nDim 2', fontsize=9)
axes[1, 0].set_ylabel('LumA / LumB\n\nDim 2', fontsize=9)
 
fig.savefig(OUTPUT_DIR / 'figS1_embedding_overlap.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS1_embedding_overlap.png', dpi=200)
plt.show()
print('Saved Fig S1.')

In [ ]:
methods_ordered = ['K-means', 'FCM', 'WFCM']
n_methods = len(methods_ordered)
 
fig, axes = plt.subplots(n_emb, n_methods + 1, figsize=(4.5 * (n_methods + 1), 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S2 — Hard cluster assignments in alternative embeddings\n'
             'Per-subtype recall is calculated from the current run.',
             fontsize=11)
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    ax_gt = axes[row, 0]
    for s in SUBTYPES:
        mask = y_arr == s
        ax_gt.scatter(Z[mask, 0], Z[mask, 1], s=10, alpha=0.6,
                      color=SUBTYPE_COLORS[s], label=s, rasterized=True)
    ax_gt.set(title='PAM50 truth' if row == 0 else '',
              xlabel='Dim 1', ylabel=f'{emb_name}\nDim 2')
    if row == 0:
        ax_gt.legend(fontsize=7, loc='upper right')
 
    for col, method in enumerate(methods_ordered, start=1):
        ax = axes[row, col]
        mapped = evaluations[method]['membership_df']['mapped_cluster'].to_numpy()
        for s in SUBTYPES:
            mask = mapped == s
            ax.scatter(Z[mask, 0], Z[mask, 1], s=10, alpha=0.6,
                       color=SUBTYPE_COLORS[s], label=s, rasterized=True)
        recall_str = '  '.join(
            f'{s[0]}:{evaluations[method]["subtype_recall"][s]:.2f}'
            for s in SUBTYPES)
        ax.set(title=f'{method}\n{recall_str}' if row == 0 else '',
               xlabel='Dim 1')
 
fig.savefig(OUTPUT_DIR / 'figS2_cluster_assignments.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS2_cluster_assignments.png', dpi=200)
plt.show()
print('Saved Fig S2.')

In [ ]:
fig, axes = plt.subplots(n_emb, n_methods, figsize=(4.5 * n_methods, 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S3 — Membership uncertainty (1 – max membership)', fontsize=11)

uncertainty_cmap = plt.cm.plasma
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    for col, method in enumerate(methods_ordered):
        ax = axes[row, col]
        unc = evaluations[method]['membership_df']['uncertainty'].to_numpy()
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=unc, cmap=uncertainty_cmap,
                        s=10, alpha=0.8, vmin=0, vmax=1, rasterized=True)
        ax.set(title=f'{method}' if row == 0 else '',
               xlabel='Dim 1',
               ylabel=f'{emb_name}\nDim 2' if col == 0 else '')
        if col == n_methods - 1:
            cbar = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
            cbar.set_label('Uncertainty\n(1 – max membership)', fontsize=8)
 
fig.savefig(OUTPUT_DIR / 'figS3_uncertainty_embedding.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS3_uncertainty_embedding.png', dpi=200)
plt.show()
print('Saved Fig S3.')

In [ ]:
fig, axes = plt.subplots(n_emb, n_methods, figsize=(4.5 * n_methods, 4.5 * n_emb),
                         constrained_layout=True)
if axes.ndim == 1:
    axes = axes[np.newaxis, :]
 
fig.suptitle('Figure S4 — Membership in clusters mapped to LumB', fontsize=11)

lumB_cmap = plt.cm.YlOrRd
 
for row, (emb_name, Z) in enumerate(embedding_dict.items()):
    for col, method in enumerate(methods_ordered):
        ax = axes[row, col]
        mdf = evaluations[method]['membership_df']
        lumB_mem = mdf['LumB'].to_numpy()
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=lumB_mem, cmap=lumB_cmap,
                        s=10, alpha=0.8, vmin=0, vmax=1, rasterized=True)
        recall_val = evaluations[method]['subtype_recall']['LumB']
        ax.set(title=f'{method}  (LumB recall = {recall_val:.3f})' if row == 0 else '',
               xlabel='Dim 1',
               ylabel=f'{emb_name}\nDim 2' if col == 0 else '')
        true_lumB = y_arr == 'LumB'
        ax.scatter(Z[true_lumB, 0], Z[true_lumB, 1], s=22, facecolors='none',
                   edgecolors='black', linewidths=0.4, alpha=0.6, rasterized=True,
                   label='True LumB' if (row == 0 and col == 0) else '')
        if col == n_methods - 1:
            cbar = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
            cbar.set_label('LumB membership', fontsize=8)
 
axes[0, 0].legend(fontsize=7)
fig.savefig(OUTPUT_DIR / 'figS4_lumB_membership.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS4_lumB_membership.png', dpi=200)
plt.show()
print('Saved Fig S4.')

In [ ]:
# Figure S5 - Luminal recall, subtype recall, global metrics, and soft memberships
methods = ['K-means', 'FCM', 'WFCM']
method_colors = dict(zip(methods, ['#4E79A7', '#F28E2B', '#E15759']))
metrics = comparison.loc[methods]
bar_width = 0.22

fig, (ax_luminal, ax_recall, ax_global, ax_soft) = plt.subplots(
    1, 4, figsize=(18, 4.8), layout='constrained'
)
fig.suptitle('Figure S5 - Per-subtype, luminal, global, and soft-membership metrics')

# Mean recall gives LumA and LumB equal weight.
luminal_recall = metrics[['LumA_recall', 'LumB_recall']].mean(axis=1)
bars = ax_luminal.bar(methods, luminal_recall, color=list(method_colors.values()), alpha=0.9)
ax_luminal.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax_luminal.axhline(0.80, color='grey', linewidth=0.8, linestyle='--')
ax_luminal.set(title='Luminal recall\n(LumA + LumB) / 2', ylabel='Luminal recall',
               ylim=(0, 1.05))

subtype_x = np.arange(len(SUBTYPES))
lum_a, lum_b = SUBTYPES.index('LumA'), SUBTYPES.index('LumB')
ax_recall.axvspan(lum_a - 0.45, lum_b + 0.45, color='#F2CF5B', alpha=0.17)
ax_recall.axhline(0.80, color='grey', linewidth=0.8, linestyle='--',
                  label='0.80 reference')
for i, method in enumerate(methods):
    ax_recall.bar(subtype_x + (i - 1) * bar_width,
                  metrics.loc[method, [f'{s}_recall' for s in SUBTYPES]],
                  bar_width, label=method, color=method_colors[method], alpha=0.9)
ax_recall.set(xticks=subtype_x, xticklabels=SUBTYPES, ylabel='Hard recall',
              title='Per-subtype recall', ylim=(0, 1.26))
ax_recall.legend(fontsize=8, loc='upper center', ncol=2)
for tick, subtype in zip(ax_recall.get_xticklabels(), SUBTYPES):
    tick.set_color(SUBTYPE_COLORS[subtype])

wfcm_lum_b = metrics.at['WFCM', 'LumB_recall']
lum_b_difference = wfcm_lum_b - metrics.at['K-means', 'LumB_recall']
ax_recall.annotate(f'{lum_b_difference:+.2f} vs K-means',
                   xy=(lum_b + bar_width, wfcm_lum_b),
                   xytext=(lum_b - 0.1, wfcm_lum_b + 0.14), ha='right',
                   fontsize=8, color=method_colors['WFCM'],
                   arrowprops=dict(arrowstyle='->', color=method_colors['WFCM']))

metric_x = np.arange(2)
for i, method in enumerate(methods):
    ax_global.bar(metric_x + (i - 1) * bar_width, metrics.loc[method, ['ARI', 'NMI']],
                  bar_width, label=method, color=method_colors[method], alpha=0.9)
ax_global.set(xticks=metric_x, xticklabels=['ARI', 'NMI'], ylabel='Score',
              title='Global clustering metrics',
              ylim=(min(0, metrics['ARI'].min() - 0.03), 1.05))
ax_global.legend(fontsize=8)

for offset, method in zip([-bar_width / 2, bar_width / 2], ['FCM', 'WFCM']):
    scores = [metrics.at[method, 'mean_max_membership'],
               metrics.at[method, 'mean_entropy_normalized']]
    bars = ax_soft.bar(metric_x + offset, scores, bar_width, label=method,
                       color=method_colors[method], alpha=0.9)
    ax_soft.bar_label(bars, fmt='%.2f', padding=3, fontsize=8)
ax_soft.set(xticks=metric_x, ylabel='Score', ylim=(0, 1.05),
            xticklabels=['Mean maximum\nmembership',
                         'Normalized entropy'],
            title='Soft membership quality\n(FCM vs WFCM)')
ax_soft.legend(fontsize=8)

fig.savefig(OUTPUT_DIR / 'figS5_recall_metrics.pdf', dpi=200)
fig.savefig(OUTPUT_DIR / 'figS5_recall_metrics.png', dpi=200)
plt.show()
print('Saved Fig S5.')


## Appendix H.4. Sensitivity to the number of retained PCs

Set **RUN_PC_SENSITIVITY = True** in the settings cell. Supply the STAR TPM matrix
(EXPR_PATH) and matching clinical annotations (CLINICAL_SOURCE), even when the main
analysis uses the prepared CSV. The sweep verifies the cohort, label order, and
reconstructed five-PC scores. It reuses the primary five-PC fit and refits all
three methods at 3, 10, and 20 PCs using the same seeds and selection rules.
For the 3-, 10-, and 20-PC WFCM refits, the fitting and validation clouds
each use PC_SENSITIVITY_NUM_IS scrambled Sobol points with separate seeds.
The same independent NLL accuracy tolerance and ranking check apply. Failed fits or checks are recorded with a status and blank
performance metrics.

The sweep uses point estimates and leaves the primary bootstrap in the five-PC
analysis. The table and diagnostics are saved under **results_breast/pc_sensitivity/**.


In [ ]:
def fit_pc_sensitivity(scores, labels, sweep_dir, n_pc):
    num_is = 2**19 if n_pc == 20 else PC_SENSITIVITY_NUM_IS
    km = KMeans(n_clusters=K, n_init=100, random_state=123).fit(scores)
    km_fit = dict(U=np.eye(K)[km.labels_], centers=km.cluster_centers_,
                  converged=bool(km.n_iter_ < km.max_iter))
    summaries = {'K-means': evaluate_method(km_fit, labels, 'K-means')['summary']}
    try:
        fcm, fcm_search = select_fcm(scores, grid, FCM_SEEDS, num_is)
        fcm_search.to_csv(sweep_dir / f'fcm_search_{n_pc}pc.csv', index=False)
        summaries['FCM'] = evaluate_method(fcm, labels, 'FCM')['summary']
    except (RuntimeError, FloatingPointError, np.linalg.LinAlgError) as error:
        summaries['FCM'] = {'status': f'fit_failed: {error}'}

    try:
        fit, rows, _, candidates = select_m(
            scores, K, grid, num_is=num_is, proposal_components=PROPOSAL_COMPONENTS,
            proposal_seed=IS_SEED, seeds=WFCM_SEEDS, max_outer=MAX_OUTER,
            inner_maxiter=INNER_MAXITER, K_post=K_POST)
        search = pd.DataFrame(rows)
        search_path = sweep_dir / f'wfcm_search_{n_pc}pc.csv'
        search.to_csv(search_path, index=False)
        if any(not f['converged'] or f['optimizer_failures']
               or (not f['post_success'] and f['post_status'] != 1) for f in candidates):
            summaries['WFCM'] = {'status': 'candidate_fit_failed'}
            return summaries

        Z, logq, _ = build_proposal(
            scores, num_is, PROPOSAL_COMPONENTS, VALIDATION_SEED)
        search['independent_NLL'] = [
            nll_gradient(f['theta'], scores, Z, logq, f['m'], K)[0] for f in candidates
        ]
        search['NLL_difference'] = search['independent_NLL'] - search['NLL']
        search.to_csv(search_path, index=False)
        if (not np.isfinite(search['independent_NLL']).all()
                or (search['NLL_difference'].abs() > NORMALIZER_NLL_TOL).any()):
            summaries['WFCM'] = {'status': 'independent_NLL_accuracy_failed'}
        elif search.loc[search['independent_NLL'].idxmin(), 'm'] != fit['m']:
            summaries['WFCM'] = {'status': 'independent_m_ranking_changed'}
        else:
            summaries['WFCM'] = evaluate_method(fit, labels, 'WFCM')['summary']
    except (RuntimeError, FloatingPointError, np.linalg.LinAlgError) as error:
        summaries['WFCM'] = {'status': f'fit_failed: {error}'}
    return summaries


def run_pc_sensitivity():
    pc_grid = (3, 5, 10, 20)
    sweep_dir = OUTPUT_DIR / 'pc_sensitivity'
    sweep_dir.mkdir(parents=True, exist_ok=True)
    hvg, sweep_labels, _, _ = load_raw_features(EXPR_PATH, CLINICAL_SOURCE, sweep_dir)
    if not sweep_labels.equals(labels):
        raise ValueError('Raw and primary analyses have different sample IDs, order, or PAM50 labels.')
    if min(hvg.shape) < max(pc_grid):
        raise ValueError('Raw expression input has too few samples or genes for the 20-PC sweep.')
    features = pca_features(hvg, STANDARDIZE_GENES)
    reference_pca = PCA(n_components=N_PC, random_state=123)
    reference_scores = reference_pca.fit_transform(features)
    signs = np.where((reference_scores * X_pca).sum(axis=0) < 0, -1, 1)
    if not np.allclose(reference_scores * signs, X_pca, rtol=1e-7, atol=1e-7):
        raise ValueError('Raw inputs do not reproduce the primary PC scores. Check input provenance.')

    rows = []
    for n_pc in pc_grid:
        print(f'PC sensitivity: {n_pc} components', flush=True)
        if n_pc == N_PC:
            sweep_pca = reference_pca
            summaries = comparison.to_dict(orient='index')
        else:
            sweep_pca = PCA(n_components=n_pc, random_state=123)
            scores = sweep_pca.fit_transform(features)
            summaries = fit_pc_sensitivity(scores, sweep_labels, sweep_dir, n_pc)

        for method, summary in summaries.items():
            rows.append({
                'N_PC': n_pc,
                'variance_explained_pct': 100 * sweep_pca.explained_variance_ratio_.sum(),
                'method': method, 'm': summary.get('m', np.nan),
                'LumB_recall': summary.get('LumB_recall', np.nan),
                'mean_luminal_recall': (summary.get('LumA_recall', np.nan)
                                        + summary.get('LumB_recall', np.nan)) / 2,
                'ARI': summary.get('ARI', np.nan), 'NMI': summary.get('NMI', np.nan),
                'status': summary.get('status', 'ok' if summary.get('converged') else 'not_converged'),
            })
        table = pd.DataFrame(rows)
        table.to_csv(sweep_dir / 'pca_sensitivity.csv', index=False)
    print('Saved PC sensitivity results to', sweep_dir)
    return table

if RUN_PC_SENSITIVITY:
    pc_sensitivity = run_pc_sensitivity()
    display(pc_sensitivity.round(3))
